lost the link

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [9]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))
device=torch.device('cuda')

True
1
NVIDIA GeForce GTX 1050


### train only

In [11]:
# rasmeri uzhe odinakovie
transform =transforms.Compose([transforms.ToTensor()])
dataset = datasets.ImageFolder('../data/cats_vs_dogs/archive/dataset', transform=transform)
loader=DataLoader(dataset, batch_size=8, shuffle=True)
# 64x64x3 -> 32x32x16 -> 16x16x32 -> 8x8x64 -> 4x4x64 1x16x16x32 -> 128 -> binary class. out.
simp_cnn=nn.Sequential(
    nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(1),
    nn.Conv2d(64,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64*8*8, 128), nn.ReLU(), nn.Linear(128,2)
)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(simp_cnn.parameters(), lr=0.001)
simp_cnn.to(device)
for e in range(10):
    for imgs, lbls in loader:
        imgs=imgs.to(device)
        lbls=lbls.to(device)
        preds=simp_cnn(imgs)
        loss=criterion(preds, lbls)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(e,', loss:', loss.item())

0 , loss: 0.6495633125305176
1 , loss: 0.5116482377052307
2 , loss: 0.722319483757019
3 , loss: 0.48931580781936646
4 , loss: 0.7918381094932556
5 , loss: 0.16257789731025696
6 , loss: 0.2364865094423294
7 , loss: 0.18495041131973267
8 , loss: 0.041055068373680115
9 , loss: 0.05107318237423897


In [13]:
# models.shufflenet_v2_x0_5, models.mobilenet_v2(width_mult=0.35)

sqnet=models.squeezenet1_1(weights='DEFAULT')
sqnet.classifier[1]=torch.nn.Conv2d(512, 2, kernel_size=1)
sqnet.num_classes=2
sqnet.to(device)
critSq=nn.CrossEntropyLoss()
optimSq=torch.optim.Adam(sqnet.parameters(), lr=0.001)
for e in range(10):
    for imgs, lbls in loader:
        imgs=imgs.to(device)
        lbls=lbls.to(device)
        preds=sqnet(imgs)
        loss=critSq(preds, lbls)
        optimSq.zero_grad()
        loss.backward()
        optimSq.step()
    print(e,', loss:', loss.item())

0 , loss: 0.6931471824645996
1 , loss: 0.6931471824645996
2 , loss: 0.6931471824645996
3 , loss: 0.6931471824645996
4 , loss: 0.6931471824645996
5 , loss: 0.6931471824645996
6 , loss: 0.6931471824645996
7 , loss: 0.6931471824645996
8 , loss: 0.6931471824645996
9 , loss: 0.6931471824645996


In [17]:
# models.shufflenet_v2_x0_5, models.mobilenet_v2(width_mult=0.35)

shnet=models.shufflenet_v2_x0_5(weights='DEFAULT')
shnet.fc=torch.nn.Linear(shnet.fc.in_features, 2)
shnet.to(device)
critSh=nn.CrossEntropyLoss()
optimSh=torch.optim.Adam(simp_cnn.parameters(), lr=0.001)
for e in range(10):
    for imgs, lbls in loader:
        imgs=imgs.to(device)
        lbls=lbls.to(device)
        preds=shnet(imgs)
        loss=critSh(preds, lbls)
        optimSh.zero_grad()
        loss.backward()
        optimSh.step()
    print(e,', loss:', loss.item())

0 , loss: 0.7049892544746399
1 , loss: 0.6918444633483887
2 , loss: 0.6914681196212769
3 , loss: 0.7055774927139282
4 , loss: 0.6948453783988953
5 , loss: 0.6970992088317871
6 , loss: 0.6825988292694092
7 , loss: 0.6890841126441956
8 , loss: 0.6848611831665039
9 , loss: 0.6951489448547363


In [20]:
# models.shufflenet_v2_x0_5, models.mobilenet_v2(width_mult=0.35)

mobnet = models.mobilenet_v2(width_mult=0.35)
mobnet.classifier[1] = nn.Linear(mobnet.classifier[1].in_features,2)
mobnet.to(device)

critMob = nn.CrossEntropyLoss()
optimMob = torch.optim.Adam(mobnet.parameters(), lr=0.001)

for e in range(10):
    for imgs, lbls in loader:
        imgs = imgs.to(device)
        lbls = lbls.to(device)
        preds = mobnet(imgs)
        loss = critMob(preds, lbls)
        optimMob.zero_grad()
        loss.backward()
        optimMob.step()

    print(e, "loss:", loss.item())

0 loss: 0.4953073561191559
1 loss: 0.8378094434738159
2 loss: 0.6019964218139648
3 loss: 0.5926432609558105
4 loss: 0.6173654198646545
5 loss: 0.7626391053199768
6 loss: 0.47930532693862915
7 loss: 0.34521204233169556
8 loss: 0.18932199478149414
9 loss: 0.37548109889030457
